In [7]:
# ============================================================
# RESTAURANT MENU & ORDER MANAGEMENT SYSTEM
# ============================================================

# ── Provided Data ──────────────────────────────────────────

menu = {
    "Paneer Tikka":   {"category": "Starters",  "price": 180.0, "available": True},
    "Chicken Wings":  {"category": "Starters",  "price": 220.0, "available": False},
    "Veg Soup":       {"category": "Starters",  "price": 120.0, "available": True},
    "Butter Chicken": {"category": "Mains",     "price": 320.0, "available": True},
    "Dal Tadka":      {"category": "Mains",     "price": 180.0, "available": True},
    "Veg Biryani":    {"category": "Mains",     "price": 250.0, "available": True},
    "Garlic Naan":    {"category": "Mains",     "price":  40.0, "available": True},
    "Gulab Jamun":    {"category": "Desserts",  "price":  90.0, "available": True},
    "Rasgulla":       {"category": "Desserts",  "price":  80.0, "available": True},
    "Ice Cream":      {"category": "Desserts",  "price": 110.0, "available": False},
}

inventory = {
    "Paneer Tikka":   {"stock": 10, "reorder_level": 3},
    "Chicken Wings":  {"stock":  8, "reorder_level": 2},
    "Veg Soup":       {"stock": 15, "reorder_level": 5},
    "Butter Chicken": {"stock": 12, "reorder_level": 4},
    "Dal Tadka":      {"stock": 20, "reorder_level": 5},
    "Veg Biryani":    {"stock":  6, "reorder_level": 3},
    "Garlic Naan":    {"stock": 30, "reorder_level": 10},
    "Gulab Jamun":    {"stock":  5, "reorder_level": 2},
    "Rasgulla":       {"stock":  4, "reorder_level": 3},
    "Ice Cream":      {"stock":  7, "reorder_level": 4},
}

sales_log = {
    "2025-01-01": [
        {"order_id": 1,  "items": ["Paneer Tikka", "Garlic Naan"],          "total": 220.0},
        {"order_id": 2,  "items": ["Gulab Jamun", "Veg Soup"],              "total": 210.0},
        {"order_id": 3,  "items": ["Butter Chicken", "Garlic Naan"],        "total": 360.0},
    ],
    "2025-01-02": [
        {"order_id": 4,  "items": ["Dal Tadka", "Garlic Naan"],             "total": 220.0},
        {"order_id": 5,  "items": ["Veg Biryani", "Gulab Jamun"],           "total": 340.0},
    ],
    "2025-01-03": [
        {"order_id": 6,  "items": ["Paneer Tikka", "Rasgulla"],             "total": 260.0},
        {"order_id": 7,  "items": ["Butter Chicken", "Veg Biryani"],        "total": 570.0},
        {"order_id": 8,  "items": ["Garlic Naan", "Gulab Jamun"],           "total": 130.0},
    ],
    "2025-01-04": [
        {"order_id": 9,  "items": ["Dal Tadka", "Garlic Naan", "Rasgulla"], "total": 300.0},
        {"order_id": 10, "items": ["Paneer Tikka", "Gulab Jamun"],          "total": 270.0},
    ],
}


# ============================================================
# TASK 1 — Explore the Menu
# ============================================================

# ── Part A: Print menu grouped by category ─────────────────

# First, collect all unique categories (keeping order)
categories = []
for item in menu.values():
    if item["category"] not in categories:
        categories.append(item["category"])

# Now print each category and its items
for category in categories:
    print(f"{'='*5} {category} {'='*5}")
    for item_name, details in menu.items():
        if details["category"] == category:
            status = "[Available]" if details["available"] else "[Unavailable]"
            # :-16s pads the name to 16 characters so columns line up neatly
            print(f"  {item_name:<16} ₹{details['price']:.2f}   {status}")

# ── Part B: Menu Statistics ─────────────────────────────────

print("" + "="*40)
print("MENU STATISTICS")
print("="*40)

# Total number of items
total_items = len(menu)
print(f"Total items on menu     : {total_items}")

# Total available items
available_items = sum(1 for details in menu.values() if details["available"])
print(f"Total available items   : {available_items}")

# Most expensive item — compare prices and track the winner
most_expensive_name = None
most_expensive_price = 0

for item_name, details in menu.items():
    if details["price"] > most_expensive_price:
        most_expensive_price = details["price"]
        most_expensive_name  = item_name

print(f"Most expensive item     : {most_expensive_name} (₹{most_expensive_price:.2f})")

# Items priced under ₹150
print("Items priced under ₹150:")
for item_name, details in menu.items():
    if details["price"] < 150:
        print(f"  {item_name:<16} ₹{details['price']:.2f}")


===== Starters =====
  Paneer Tikka     ₹180.00   [Available]
  Chicken Wings    ₹220.00   [Unavailable]
  Veg Soup         ₹120.00   [Available]
===== Mains =====
  Butter Chicken   ₹320.00   [Available]
  Dal Tadka        ₹180.00   [Available]
  Veg Biryani      ₹250.00   [Available]
  Garlic Naan      ₹40.00   [Available]
===== Desserts =====
  Gulab Jamun      ₹90.00   [Available]
  Rasgulla         ₹80.00   [Available]
  Ice Cream        ₹110.00   [Unavailable]
MENU STATISTICS
Total items on menu     : 10
Total available items   : 8
Most expensive item     : Butter Chicken (₹320.00)
Items priced under ₹150:
  Veg Soup         ₹120.00
  Garlic Naan      ₹40.00
  Gulab Jamun      ₹90.00
  Rasgulla         ₹80.00
  Ice Cream        ₹110.00


In [8]:
# ============================================================
# TASK 2 — Cart Operations
# ============================================================

cart = []   # Each entry: {"item": "...", "quantity": N, "price": X.X}

# ── Helper Function: Add item to cart ──────────────────────
def add_to_cart(item_name, quantity):
    # Check 1: Does the item exist in the menu?
    if item_name not in menu:
        print(f"  ❌ '{item_name}' does not exist on the menu.")
        return

    # Check 2: Is the item available?
    if not menu[item_name]["available"]:
        print(f"  ❌ '{item_name}' is currently unavailable.")
        return

    # Check 3: Is the item already in the cart?
    for entry in cart:
        if entry["item"] == item_name:
            entry["quantity"] += quantity
            print(f"  ✅ Updated '{item_name}' quantity to {entry['quantity']}.")
            return

    # Item is new — add a fresh entry
    cart.append({
        "item":     item_name,
        "quantity": quantity,
        "price":    menu[item_name]["price"]
    })
    print(f"  ✅ Added '{item_name}' × {quantity} to cart.")


# ── Helper Function: Remove item from cart ─────────────────
def remove_from_cart(item_name):
    for entry in cart:
        if entry["item"] == item_name:
            cart.remove(entry)
            print(f"  🗑️  Removed '{item_name}' from cart.")
            return
    print(f"  ⚠️  '{item_name}' is not in the cart.")


# ── Helper Function: Update quantity ───────────────────────
def update_quantity(item_name, new_quantity):
    for entry in cart:
        if entry["item"] == item_name:
            entry["quantity"] = new_quantity
            print(f"  🔄 Updated '{item_name}' quantity to {new_quantity}.")
            return
    print(f"  ⚠️  '{item_name}' is not in the cart.")


# ── Helper Function: Print current cart state ──────────────
def print_cart():
    if not cart:
        print("  🛒 Cart is empty.")
    else:
        print("  Current Cart:")
        for entry in cart:
            line_total = entry["quantity"] * entry["price"]
            print(f"    {entry['item']:<16} x{entry['quantity']}  ₹{line_total:.2f}")


# ── Simulate the order sequence ────────────────────────────

print("="*40)
print("STEP 1: Add Paneer Tikka × 2")
print("="*40)
add_to_cart("Paneer Tikka", 2)
print_cart()

print("" + "="*40)
print("STEP 2: Add Gulab Jamun × 1")
print("="*40)
add_to_cart("Gulab Jamun", 1)
print_cart()

print("" + "="*40)
print("STEP 3: Add Paneer Tikka × 1 (should update to 3)")
print("="*40)
add_to_cart("Paneer Tikka", 1)
print_cart()

print("" + "="*40)
print("STEP 4: Try to add 'Mystery Burger' (not on menu)")
print("="*40)
add_to_cart("Mystery Burger", 1)
print_cart()

print("" + "="*40)
print("STEP 5: Try to add 'Chicken Wings' (unavailable)")
print("="*40)
add_to_cart("Chicken Wings", 1)
print_cart()

print("" + "="*40)
print("STEP 6: Remove Gulab Jamun")
print("="*40)
remove_from_cart("Gulab Jamun")
print_cart()

# ── Final Order Summary ─────────────────────────────────────

print("" + "="*36)
print("       ORDER SUMMARY")
print("="*36)

subtotal = 0
for entry in cart:
    line_total = entry["quantity"] * entry["price"]
    subtotal  += line_total
    print(f"  {entry['item']:<16} x{entry['quantity']}   ₹{line_total:.2f}")

gst           = subtotal * 0.05
total_payable = subtotal + gst

print("-"*36)
print(f"  {'Subtotal:':<24} ₹{subtotal:.2f}")
print(f"  {'GST (5%):':<24} ₹{gst:.2f}")
print(f"  {'Total Payable:':<24} ₹{total_payable:.2f}")
print("="*36)

STEP 1: Add Paneer Tikka × 2
  ✅ Added 'Paneer Tikka' × 2 to cart.
  Current Cart:
    Paneer Tikka     x2  ₹360.00
STEP 2: Add Gulab Jamun × 1
  ✅ Added 'Gulab Jamun' × 1 to cart.
  Current Cart:
    Paneer Tikka     x2  ₹360.00
    Gulab Jamun      x1  ₹90.00
STEP 3: Add Paneer Tikka × 1 (should update to 3)
  ✅ Updated 'Paneer Tikka' quantity to 3.
  Current Cart:
    Paneer Tikka     x3  ₹540.00
    Gulab Jamun      x1  ₹90.00
STEP 4: Try to add 'Mystery Burger' (not on menu)
  ❌ 'Mystery Burger' does not exist on the menu.
  Current Cart:
    Paneer Tikka     x3  ₹540.00
    Gulab Jamun      x1  ₹90.00
STEP 5: Try to add 'Chicken Wings' (unavailable)
  ❌ 'Chicken Wings' is currently unavailable.
  Current Cart:
    Paneer Tikka     x3  ₹540.00
    Gulab Jamun      x1  ₹90.00
STEP 6: Remove Gulab Jamun
  🗑️  Removed 'Gulab Jamun' from cart.
  Current Cart:
    Paneer Tikka     x3  ₹540.00
       ORDER SUMMARY
  Paneer Tikka     x3   ₹540.00
------------------------------------
  Su

In [9]:
# ============================================================
# TASK 3 — Inventory Tracker with Deep Copy
# ============================================================

import copy

# ── Step 1: Deep copy inventory before making any changes ──
# A deep copy means changes to 'inventory' won't affect 'inventory_backup'
inventory_backup = copy.deepcopy(inventory)

# ── Step 2: Demonstrate deep copy works ────────────────────
print("DEMO: Changing Paneer Tikka stock in inventory...")
inventory["Paneer Tikka"]["stock"] = 999   # temporary change

print(f"  inventory['Paneer Tikka']['stock']        = {inventory['Paneer Tikka']['stock']}")
print(f"  inventory_backup['Paneer Tikka']['stock'] = {inventory_backup['Paneer Tikka']['stock']}")
print("  Backup is unaffected — deep copy works!")

# Restore inventory to original state before continuing
inventory["Paneer Tikka"]["stock"] = 10
print("  Inventory restored to original.")

# ── Step 3: Deduct cart quantities from inventory ──────────
print("="*40)
print("ORDER FULFILMENT — Deducting from Inventory")
print("="*40)

for entry in cart:
    item_name = entry["item"]
    qty_needed = entry["quantity"]
    current_stock = inventory[item_name]["stock"]

    if current_stock >= qty_needed:
        inventory[item_name]["stock"] -= qty_needed
        print(f"   {item_name}: Deducted {qty_needed}. Remaining stock: {inventory[item_name]['stock']}")
    else:
        # Not enough stock — deduct only what's available
        print(f"   {item_name}: Only {current_stock} in stock, needed {qty_needed}. Deducting {current_stock}.")
        inventory[item_name]["stock"] = 0

# ── Step 4: Reorder Alerts ─────────────────────────────────
print("" + "="*40)
print("REORDER ALERTS")
print("="*40)

alert_found = False
for item_name, details in inventory.items():
    if details["stock"] <= details["reorder_level"]:
        print(f"  ⚠ Reorder Alert: {item_name} — Only {details['stock']} unit(s) left "
              f"(reorder level: {details['reorder_level']})")
        alert_found = True

if not alert_found:
    print("  All items are sufficiently stocked.")

# ── Step 5: Compare inventory vs backup ────────────────────
print("" + "="*40)
print("INVENTORY vs BACKUP COMPARISON")
print("="*40)
print(f"  {'Item':<16} {'Current Stock':>14} {'Backup Stock':>13}")
print("-"*45)
for item_name in inventory:
    curr = inventory[item_name]["stock"]
    back = inventory_backup[item_name]["stock"]
    diff = "← changed" if curr != back else ""
    print(f"  {item_name:<16} {curr:>14} {back:>13}  {diff}")


DEMO: Changing Paneer Tikka stock in inventory...
  inventory['Paneer Tikka']['stock']        = 999
  inventory_backup['Paneer Tikka']['stock'] = 10
  Backup is unaffected — deep copy works!
  Inventory restored to original.
ORDER FULFILMENT — Deducting from Inventory
   Paneer Tikka: Deducted 3. Remaining stock: 7
REORDER ALERTS
  All items are sufficiently stocked.
INVENTORY vs BACKUP COMPARISON
  Item              Current Stock  Backup Stock
---------------------------------------------
  Paneer Tikka                  7            10  ← changed
  Chicken Wings                 8             8  
  Veg Soup                     15            15  
  Butter Chicken               12            12  
  Dal Tadka                    20            20  
  Veg Biryani                   6             6  
  Garlic Naan                  30            30  
  Gulab Jamun                   5             5  
  Rasgulla                      4             4  
  Ice Cream                     7             

In [10]:
# ============================================================
# TASK 4 — Daily Sales Log Analysis
# ============================================================

# ── Part A: Total revenue per day ──────────────────────────
print("="*40)
print("REVENUE PER DAY")
print("="*40)

daily_revenue = {}   # will store {date: total_revenue}

for date, orders in sales_log.items():
    day_total = sum(order["total"] for order in orders)
    daily_revenue[date] = day_total
    print(f"  {date}  →  ₹{day_total:.2f}")

# ── Part B: Best-selling day ───────────────────────────────
best_day         = None
best_day_revenue = 0

for date, revenue in daily_revenue.items():
    if revenue > best_day_revenue:
        best_day_revenue = revenue
        best_day         = date

print(f" Best-selling day: {best_day} with ₹{best_day_revenue:.2f} in revenue")

# ── Part C: Most ordered item ──────────────────────────────
# Count how many individual orders each item appears in
item_order_count = {}   # {item_name: number_of_orders_it_appeared_in}

for date, orders in sales_log.items():
    for order in orders:
        for item in order["items"]:
            if item not in item_order_count:
                item_order_count[item] = 0
            item_order_count[item] += 1

# Find the item with the highest count
most_ordered_item  = None
most_ordered_count = 0

for item_name, count in item_order_count.items():
    if count > most_ordered_count:
        most_ordered_count = count
        most_ordered_item  = item_name

print(f" Most ordered item: {most_ordered_item} (appeared in {most_ordered_count} orders)")

# ── Part D: Add new day and reprint stats ──────────────────
sales_log["2025-01-05"] = [
    {"order_id": 11, "items": ["Butter Chicken", "Gulab Jamun", "Garlic Naan"], "total": 490.0},
    {"order_id": 12, "items": ["Paneer Tikka", "Rasgulla"],                     "total": 260.0},
]

print("" + "="*40)
print("UPDATED REVENUE PER DAY (after adding 2025-01-05)")
print("="*40)

daily_revenue = {}   # recalculate with the new day included

for date, orders in sales_log.items():
    day_total = sum(order["total"] for order in orders)
    daily_revenue[date] = day_total
    print(f"  {date}  →  ₹{day_total:.2f}")

# Recalculate best-selling day
best_day         = None
best_day_revenue = 0

for date, revenue in daily_revenue.items():
    if revenue > best_day_revenue:
        best_day_revenue = revenue
        best_day         = date

print(f"Updated Best-selling day: {best_day} with ₹{best_day_revenue:.2f} in revenue")

# ── Part E: Numbered list of all orders using enumerate ────
print("" + "="*55)
print("ALL ORDERS (NUMBERED)")
print("="*55)

all_orders = []   # collect (date, order) pairs first
for date, orders in sales_log.items():
    for order in orders:
        all_orders.append((date, order))

# enumerate starts counting from 1
for serial_no, (date, order) in enumerate(all_orders, start=1):
    items_str = ", ".join(order["items"])
    print(f"  {serial_no:<3} [{date}] Order #{order['order_id']:<3}"
          f" — ₹{order['total']:.2f} — Items: {items_str}")


REVENUE PER DAY
  2025-01-01  →  ₹790.00
  2025-01-02  →  ₹560.00
  2025-01-03  →  ₹960.00
  2025-01-04  →  ₹570.00
 Best-selling day: 2025-01-03 with ₹960.00 in revenue
 Most ordered item: Garlic Naan (appeared in 5 orders)
UPDATED REVENUE PER DAY (after adding 2025-01-05)
  2025-01-01  →  ₹790.00
  2025-01-02  →  ₹560.00
  2025-01-03  →  ₹960.00
  2025-01-04  →  ₹570.00
  2025-01-05  →  ₹750.00
Updated Best-selling day: 2025-01-03 with ₹960.00 in revenue
ALL ORDERS (NUMBERED)
  1   [2025-01-01] Order #1   — ₹220.00 — Items: Paneer Tikka, Garlic Naan
  2   [2025-01-01] Order #2   — ₹210.00 — Items: Gulab Jamun, Veg Soup
  3   [2025-01-01] Order #3   — ₹360.00 — Items: Butter Chicken, Garlic Naan
  4   [2025-01-02] Order #4   — ₹220.00 — Items: Dal Tadka, Garlic Naan
  5   [2025-01-02] Order #5   — ₹340.00 — Items: Veg Biryani, Gulab Jamun
  6   [2025-01-03] Order #6   — ₹260.00 — Items: Paneer Tikka, Rasgulla
  7   [2025-01-03] Order #7   — ₹570.00 — Items: Butter Chicken, Veg Biryani